In [1]:
import ee
import geemap
geemap.ee_initialize()

Enter verification code:  4/1AfrIepDTsQZrWBINAvngFhxRbTDunZztJInf9EiJmo6wxpfU1-nCW4uEO0w



Successfully saved authorization token.


In [2]:
import csv
import numpy as np
import pandas as pd
import os
import sys

In [16]:
def extract_timeseries_columns():
    # ==========================================
    # 1. CONFIGURATION
    # ==========================================
    input_csv = 'Sites.csv'
    output_csv = 'extracted_years_data.csv'
    
    lat_col = 'Latitude'
    lon_col = 'Longitude'
    
    # Years to process
    start_year = 1997
    end_year = 2024
    
    # Raster Settings
    target_scale = 4638.3       
    target_crs = 'EPSG:4326' 
    
    # Bands to extract
    bands = ['avg_rad', 'cf_cvg'] 

    # ==========================================
    # 2. READ CSV & UPLOAD POINTS
    # ==========================================
    print(f"Reading {input_csv}...")
    try:
        df = pd.read_csv(input_csv)
    except FileNotFoundError:
        print(f"Error: File {input_csv} not found.")
        sys.exit()

    features = []
    for index, row in df.iterrows():
        try:
            geom = ee.Geometry.Point([row[lon_col], row[lat_col]])
            feat = ee.Feature(geom, {'row_id': index})
            features.append(feat)
        except Exception:
            continue

    pts_collection = ee.FeatureCollection(features)
    roi = pts_collection.geometry().bounds()

    # ==========================================
    # 3. PREPARE COLLECTION
    # ==========================================
    # Load collection and select bands upfront
    raw_collection = ee.ImageCollection('NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG') \
        .filterBounds(roi) \
        .select(bands)

    if raw_collection.size().getInfo() == 0:
        print("No images found in this region.")
        sys.exit()

    # Capture Native Projection
    native_proj = raw_collection.first().projection()

    # ==========================================
    # 4. LOOP OVER YEARS & RE-RASTERIZE
    # ==========================================
    print(f"Processing years {start_year} to {end_year}...")
    
    yearly_images = []
    
    for year in range(start_year, end_year + 1):
        # Create Composite
        year_img = raw_collection.filterDate(f'{year}-01-01', f'{year}-12-31').mean()
        
        # Re-rasterize
        processed = year_img \
            .setDefaultProjection(native_proj) \
            .reduceResolution(
                reducer=ee.Reducer.mean(), 
                maxPixels=1024
            ) \
            .reproject(
                crs=target_crs, 
                scale=target_scale
            )
            
        # --- THE FIX ---
        # Instead of .select(), we use .rename()
        # This takes the existing bands (B4, B8) and renames them directly
        new_names = [f"{b}_{year}" for b in bands]
        processed = processed.rename(new_names)
        
        # Add to list
        yearly_images.append(processed)

    # ==========================================
    # 5. STACK AND EXTRACT
    # ==========================================
    # Combine all yearly images into ONE image
    final_stack = ee.Image.cat(yearly_images)
    
    print("Extracting data (this may take a moment)...")
    
    data = final_stack.reduceRegions(
        collection=pts_collection,
        reducer=ee.Reducer.first(),
        scale=target_scale
    )
    
    results = data.getInfo()
    
    # ==========================================
    # 6. PROCESS & SAVE
    # ==========================================
    extracted_dict = {}
    
    for f in results['features']:
        props = f['properties']
        if 'row_id' in props:
            idx = int(props['row_id'])
            # Exclude row_id from data
            values = {k: v for k, v in props.items() if k != 'row_id'}
            extracted_dict[idx] = values

    extracted_df = pd.DataFrame.from_dict(extracted_dict, orient='index')
    
    # Sort columns
    extracted_df = extracted_df.reindex(sorted(extracted_df.columns), axis=1)

    # Join with original
    final_df = df.join(extracted_df)
    
    final_df.to_csv(output_csv, index=False)
    print(f"Success! Data saved to: {output_csv}")
    print(final_df.head())

if __name__ == "__main__":
    extract_timeseries_columns()

Reading Sites.csv...
Processing years 1997 to 2024...
Extracting data (this may take a moment)...
Success! Data saved to: extracted_years_data.csv
           Farms   Latitude  Longitude  avg_rad_2014  avg_rad_2015  \
0       DE_Lewes  38.732719 -75.180519      5.978862      5.823205   
1  DE_Georgetown  38.639656 -75.458267      0.965645      0.903134   
2     DE_Concord  38.639386 -75.525808      1.236152      1.156262   
3      DE_Laurel  38.502697 -75.516772      1.095123      1.022415   
4     DE_Seaford  38.597775 -75.679592      0.795584      0.737493   

   avg_rad_2016  avg_rad_2017  avg_rad_2018  avg_rad_2019  avg_rad_2020  ...  \
0      5.438320      6.483913      6.156395      7.033847      6.652544  ...   
1      0.777370      0.978852      0.953851      1.041776      1.063959  ...   
2      0.986894      1.227104      1.165182      1.244551      1.339073  ...   
3      0.871781      1.125851      1.084319      1.123461      1.216753  ...   
4      0.606397      0.804082   

In [17]:
def extract_timeseries_columns():
    # ==========================================
    # 1. CONFIGURATION
    # ==========================================
    input_csv = 'Sites.csv'
    output_csv = 'VIIRS_resampled.csv'
    
    lat_col = 'Latitude'
    lon_col = 'Longitude'
    
    # Years to process
    start_year = 1997
    end_year = 2024
    
    # Raster Settings
    target_scale = 4638.3       
    target_crs = 'EPSG:4326' 
    
    # Bands to extract
    bands = ['avg_rad', 'cf_cvg'] 

    # ==========================================
    # 2. READ CSV & UPLOAD POINTS
    # ==========================================
    print(f"Reading {input_csv}...")
    try:
        df = pd.read_csv(input_csv)
    except FileNotFoundError:
        print(f"Error: File {input_csv} not found.")
        sys.exit()

    features = []
    for index, row in df.iterrows():
        try:
            geom = ee.Geometry.Point([row[lon_col], row[lat_col]])
            feat = ee.Feature(geom, {'row_id': index})
            features.append(feat)
        except Exception:
            continue

    pts_collection = ee.FeatureCollection(features)
    roi = pts_collection.geometry().bounds()

    # ==========================================
    # 3. PREPARE COLLECTION
    # ==========================================
    # Load collection and select bands upfront
    raw_collection = ee.ImageCollection('NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG') \
        .filterBounds(roi) \
        .select(bands)

    if raw_collection.size().getInfo() == 0:
        print("No images found in this region.")
        sys.exit()

    # Capture Native Projection
    native_proj = raw_collection.first().projection()

    # ==========================================
    # 4. LOOP OVER YEARS & RE-RASTERIZE
    # ==========================================
    print(f"Processing years {start_year} to {end_year}...")
    
    yearly_images = []
    
    for year in range(start_year, end_year + 1):
        # Create Composite
        year_img = raw_collection.filterDate(f'{year}-01-01', f'{year}-12-31').mean()
        
        # Re-rasterize
        processed = year_img \
            .setDefaultProjection(native_proj) \
            .reduceResolution(
                reducer=ee.Reducer.mean(), 
                maxPixels=1024
            ) \
            .reproject(
                crs=target_crs, 
                scale=target_scale
            )
            
        # --- THE FIX ---
        # Instead of .select(), we use .rename()
        # This takes the existing bands (B4, B8) and renames them directly
        new_names = [f"{b}_{year}" for b in bands]
        processed = processed.rename(new_names)
        
        # Add to list
        yearly_images.append(processed)

    # ==========================================
    # 5. STACK AND EXTRACT
    # ==========================================
    # Combine all yearly images into ONE image
    final_stack = ee.Image.cat(yearly_images)
    
    print("Extracting data (this may take a moment)...")
    
    data = final_stack.reduceRegions(
        collection=pts_collection,
        reducer=ee.Reducer.first(),
        scale=target_scale
    )
    
    results = data.getInfo()
    
    # ==========================================
    # 6. PROCESS & SAVE
    # ==========================================
    extracted_dict = {}
    
    for f in results['features']:
        props = f['properties']
        if 'row_id' in props:
            idx = int(props['row_id'])
            # Exclude row_id from data
            values = {k: v for k, v in props.items() if k != 'row_id'}
            extracted_dict[idx] = values

    extracted_df = pd.DataFrame.from_dict(extracted_dict, orient='index')
    
    # Sort columns
    extracted_df = extracted_df.reindex(sorted(extracted_df.columns), axis=1)

    # Join with original
    final_df = df.join(extracted_df)
    
    final_df.to_csv(output_csv, index=False)
    print(f"Success! Data saved to: {output_csv}")
    print(final_df.head())

if __name__ == "__main__":
    extract_timeseries_columns()

Reading Sites.csv...
Processing years 1997 to 2024...
Extracting data (this may take a moment)...
Success! Data saved to: VIIRS_resampled.csv
           Farms   Latitude  Longitude  avg_rad_2014  avg_rad_2015  \
0       DE_Lewes  38.732719 -75.180519      5.978862      5.823205   
1  DE_Georgetown  38.639656 -75.458267      0.965645      0.903134   
2     DE_Concord  38.639386 -75.525808      1.236152      1.156262   
3      DE_Laurel  38.502697 -75.516772      1.095123      1.022415   
4     DE_Seaford  38.597775 -75.679592      0.795584      0.737493   

   avg_rad_2016  avg_rad_2017  avg_rad_2018  avg_rad_2019  avg_rad_2020  ...  \
0      5.438320      6.483913      6.156395      7.033847      6.652544  ...   
1      0.777370      0.978852      0.953851      1.041776      1.063959  ...   
2      0.986894      1.227104      1.165182      1.244551      1.339073  ...   
3      0.871781      1.125851      1.084319      1.123461      1.216753  ...   
4      0.606397      0.804082      0.

In [20]:
def extract_timeseries_columns():
    # ==========================================
    # 1. CONFIGURATION
    # ==========================================
    input_csv = 'Sites.csv'
    output_csv = 'DMSP_resampled.csv'
    
    lat_col = 'Latitude'
    lon_col = 'Longitude'
    
    # Years to process
    start_year = 1997
    end_year = 2024
    
    # Raster Settings
    target_scale = 4638.3       
    target_crs = 'EPSG:4326' 
    
    # Bands to extract
    bands = ['b1'] 

    # ==========================================
    # 2. READ CSV & UPLOAD POINTS
    # ==========================================
    print(f"Reading {input_csv}...")
    try:
        df = pd.read_csv(input_csv)
    except FileNotFoundError:
        print(f"Error: File {input_csv} not found.")
        sys.exit()

    features = []
    for index, row in df.iterrows():
        try:
            geom = ee.Geometry.Point([row[lon_col], row[lat_col]])
            feat = ee.Feature(geom, {'row_id': index})
            features.append(feat)
        except Exception:
            continue

    pts_collection = ee.FeatureCollection(features)
    roi = pts_collection.geometry().bounds()

    # ==========================================
    # 3. PREPARE COLLECTION
    # ==========================================
    # Load collection and select bands upfront
    raw_collection = ee.ImageCollection('BNU/FGS/CCNL/v1') \
        .filterBounds(roi) \
        .select(bands)

    if raw_collection.size().getInfo() == 0:
        print("No images found in this region.")
        sys.exit()

    # Capture Native Projection
    native_proj = raw_collection.first().projection()

    # ==========================================
    # 4. LOOP OVER YEARS & RE-RASTERIZE
    # ==========================================
    print(f"Processing years {start_year} to {end_year}...")
    
    yearly_images = []
    
    for year in range(start_year, end_year + 1):
        # Create Composite
        year_img = raw_collection.filterDate(f'{year}-01-01', f'{year}-12-31').mean()
        
        # Re-rasterize
        processed = year_img \
            .setDefaultProjection(native_proj) \
            .reduceResolution(
                reducer=ee.Reducer.mean(), 
                maxPixels=1024
            ) \
            .reproject(
                crs=target_crs, 
                scale=target_scale
            )
            
        # --- THE FIX ---
        # Instead of .select(), we use .rename()
        # This takes the existing bands (B4, B8) and renames them directly
        new_names = [f"{b}_{year}" for b in bands]
        processed = processed.rename(new_names)
        
        # Add to list
        yearly_images.append(processed)

    # ==========================================
    # 5. STACK AND EXTRACT
    # ==========================================
    # Combine all yearly images into ONE image
    final_stack = ee.Image.cat(yearly_images)
    
    print("Extracting data (this may take a moment)...")
    
    data = final_stack.reduceRegions(
        collection=pts_collection,
        reducer=ee.Reducer.first(),
        scale=target_scale
    )
    
    results = data.getInfo()
    
    # ==========================================
    # 6. PROCESS & SAVE
    # ==========================================
    extracted_dict = {}
    
    for f in results['features']:
        props = f['properties']
        if 'row_id' in props:
            idx = int(props['row_id'])
            # Exclude row_id from data
            values = {k: v for k, v in props.items() if k != 'row_id'}
            extracted_dict[idx] = values

    extracted_df = pd.DataFrame.from_dict(extracted_dict, orient='index')
    
    # Sort columns
    extracted_df = extracted_df.reindex(sorted(extracted_df.columns), axis=1)

    # Join with original
    final_df = df.join(extracted_df)
    
    final_df.to_csv(output_csv, index=False)
    print(f"Success! Data saved to: {output_csv}")
    print(final_df.head())

if __name__ == "__main__":
    extract_timeseries_columns()

Reading Sites.csv...
Processing years 1997 to 2024...
Extracting data (this may take a moment)...
Success! Data saved to: DMSP_resampled.csv
           Farms   Latitude  Longitude    b1_1997    b1_1998    b1_1999  \
0       DE_Lewes  38.732719 -75.180519  31.122334  25.149584  25.056670   
1  DE_Georgetown  38.639656 -75.458267   4.791355   5.880246   6.976373   
2     DE_Concord  38.639386 -75.525808   5.841352   7.181507   7.709545   
3      DE_Laurel  38.502697 -75.516772   5.837561   7.111241   7.631220   
4     DE_Seaford  38.597775 -75.679592   5.431281   5.843587   5.513985   

     b1_2000    b1_2001    b1_2002    b1_2003    b1_2004    b1_2005  \
0  30.410823  24.215227  28.435851  30.118524  30.478276  33.518705   
1   7.253456   5.624690   5.329384   6.011902   5.307160   5.150614   
2   8.320260   6.989141   7.034630   7.279640   6.566702   6.822646   
3   8.511884   7.034293   6.684003   6.553494   6.032770   5.957167   
4   7.587043   5.257459   5.259655   5.195779   4.684

In [21]:
import ee
import pandas as pd
import sys
import calendar

# Initialize Earth Engine
try:
    ee.Initialize()
except Exception:
    ee.Authenticate()
    ee.Initialize()

def extract_monthly_data():
    # ==========================================
    # 1. CONFIGURATION
    # ==========================================
    input_csv = 'Sites.csv'
    output_csv = 'EVI_resampled.csv'
    
    lat_col = 'Latitude'
    lon_col = 'Longitude'
    
    # Time Range
    start_year = 1997
    end_year = 2024
    
    # Raster Settings
    target_scale = 4638.3       
    target_crs = 'EPSG:4326' 
    
    # Bands (e.g., Red and NIR)
    bands = ['EVI'] 

    # ==========================================
    # 2. READ CSV & UPLOAD POINTS
    # ==========================================
    print(f"Reading {input_csv}...")
    try:
        df = pd.read_csv(input_csv)
    except FileNotFoundError:
        print(f"Error: File {input_csv} not found.")
        sys.exit()

    features = []
    for index, row in df.iterrows():
        try:
            geom = ee.Geometry.Point([row[lon_col], row[lat_col]])
            feat = ee.Feature(geom, {'row_id': index})
            features.append(feat)
        except Exception:
            continue

    pts_collection = ee.FeatureCollection(features)
    roi = pts_collection.geometry().bounds()

    # ==========================================
    # 3. PREPARE COLLECTION
    # ==========================================
    # Load raw collection once to get projection
    raw_collection = ee.ImageCollection('MODIS/061/MOD13Q1') \
        .filterBounds(roi) \
        .select(bands)

    if raw_collection.size().getInfo() == 0:
        print("No images found in this region.")
        sys.exit()

    native_proj = raw_collection.first().projection()

    # ==========================================
    # 4. LOOP OVER MONTHS
    # ==========================================
    print(f"Processing Monthly Data from {start_year} to {end_year}...")
    
    monthly_images = []
    
    for year in range(start_year, end_year + 1):
        for month in range(1, 13):
            
            # Construct start and end dates for the month
            start_date = ee.Date.fromYMD(year, month, 1)
            end_date = start_date.advance(1, 'month')
            
            # Format label for printing and naming: "2022_01"
            date_label = f"{year}_{month:02d}"
            
            # Filter collection for this specific month
            month_col = raw_collection.filterDate(start_date, end_date)
            
            # CHECK: Does this month have images?
            # We use getInfo() here to prevent the script from crashing 
            # if a specific month is empty (common in winter/cloudy areas).
            count = month_col.size().getInfo()
            
            if count == 0:
                print(f"  Skipping {date_label}: No images found.")
                continue
            
            print(f"  Processing {date_label} ({count} images)...")

            # Create Composite
            month_img = month_col.mean()
            
            # Re-rasterize
            processed = month_img \
                .setDefaultProjection(native_proj) \
                .reduceResolution(
                    reducer=ee.Reducer.mean(), 
                    maxPixels=1024
                ) \
                .reproject(
                    crs=target_crs, 
                    scale=target_scale
                )
            
            # RENAME BANDS: "B4" -> "B4_2022_01"
            new_names = [f"{b}_{date_label}" for b in bands]
            processed = processed.rename(new_names)
            
            monthly_images.append(processed)

    if not monthly_images:
        print("Error: No data found for any month in the range.")
        sys.exit()

    # ==========================================
    # 5. STACK AND EXTRACT
    # ==========================================
    # Combine all monthly images into ONE giant image
    final_stack = ee.Image.cat(monthly_images)
    
    print("\nExtracting data (this takes longer for monthly data)...")
    
    # Extract
    data = final_stack.reduceRegions(
        collection=pts_collection,
        reducer=ee.Reducer.first(),
        scale=target_scale
    )
    
    results = data.getInfo()
    
    # ==========================================
    # 6. PROCESS & SAVE
    # ==========================================
    extracted_dict = {}
    
    for f in results['features']:
        props = f['properties']
        if 'row_id' in props:
            idx = int(props['row_id'])
            values = {k: v for k, v in props.items() if k != 'row_id'}
            extracted_dict[idx] = values

    extracted_df = pd.DataFrame.from_dict(extracted_dict, orient='index')
    
    # Sort columns so 2022_01 comes before 2022_02
    extracted_df = extracted_df.reindex(sorted(extracted_df.columns), axis=1)

    # Join with original
    final_df = df.join(extracted_df)
    
    final_df.to_csv(output_csv, index=False)
    print(f"Success! Data saved to: {output_csv}")
    
    # Preview
    cols = list(extracted_df.columns)
    print(f"Columns created ({len(cols)} total):")
    print(cols[:4], "...", cols[-4:])

if __name__ == "__main__":
    extract_monthly_data()

Reading Sites.csv...
Processing Monthly Data from 1997 to 2024...
  Skipping 1997_01: No images found.
  Skipping 1997_02: No images found.
  Skipping 1997_03: No images found.
  Skipping 1997_04: No images found.
  Skipping 1997_05: No images found.
  Skipping 1997_06: No images found.
  Skipping 1997_07: No images found.
  Skipping 1997_08: No images found.
  Skipping 1997_09: No images found.
  Skipping 1997_10: No images found.
  Skipping 1997_11: No images found.
  Skipping 1997_12: No images found.
  Skipping 1998_01: No images found.
  Skipping 1998_02: No images found.
  Skipping 1998_03: No images found.
  Skipping 1998_04: No images found.
  Skipping 1998_05: No images found.
  Skipping 1998_06: No images found.
  Skipping 1998_07: No images found.
  Skipping 1998_08: No images found.
  Skipping 1998_09: No images found.
  Skipping 1998_10: No images found.
  Skipping 1998_11: No images found.
  Skipping 1998_12: No images found.
  Skipping 1999_01: No images found.
  Skippin

In [3]:
pip install geedim

     ---------------------------------------- 0.0/57.7 kB ? eta -:--:--
     ------- -------------------------------- 10.2/57.7 kB ? eta -:--:--
     --------------------------------- ---- 51.2/57.7 kB 871.5 kB/s eta 0:00:01
     ---------------------------------------- 57.7/57.7 kB 1.0 MB/s eta 0:00:00
     ---------------------------------------- 0.0/82.2 kB ? eta -:--:--
     ---------------------------------------- 82.2/82.2 kB ? eta 0:00:00
   ---------------------------------------- 0.0/73.1 kB ? eta -:--:--
   ---------------------------------------- 73.1/73.1 kB 3.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/457.7 kB ? eta -:--:--
   --------------------- ----------------- 256.0/457.7 kB 16.4 MB/s eta 0:00:01
   ---------------------------------------- 457.7/457.7 kB 9.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/202.5 kB ? eta -:--:--
   ---------------------------------------- 202.5/202.5 kB ? eta 0:00:00
   ------------------------

In [8]:
region = ee.Geometry.Rectangle([-76, -74, 38, 42])

collection = ee.ImageCollection('OREGONSTATE/PRISM/AN81m')

prism_image = collection \
        .filterDate('2020-01-01', '2020-02-01') \
        .select('ppt') \
        .first()

image_clipped = prism_image.clip(region)

geemap.download_ee_image(image_clipped, region=region, filename='prism_ppt_2020.tif')


202001:   0%|                                                                                     |0/14 tiles …